<a id="inicio"></a>
    # AED 05 · Matrices, árboles de decisión y valor de la información

    **Núcleo U2 · S08 · RA2 / CE2.3–2.4; RA3 / CE3.2** · Colección docente · Edición 2026-10-08.

    Ejecute todas las celdas en orden con el entorno documentado en `../reproducibilidad/README.md`.
    Los controles necesitan kernel activo; las salidas guardadas permiten lectura. Los datos son
    casos docentes locales; las simulaciones y parámetros económicos se identifican como supuestos.
    Consulte la [guía maestra](../guia_maestra_aed.html), el
    [manual científico](../material_propio/AED_manual_cientifico.pdf) y la
    [matriz curricular](../COBERTURA.md). Los ejercicios son formativos.

### Antes de ejecutar

Núcleo · 90–120 min orientativos.

**Objetivo:** Comparar acciones bajo riesgo y valorar información antes de pagar por ella.

**Preparación:** Probabilidades y valor esperado; distinguir acción de estado del mundo.

**Ejemplo de referencia:** Con p alto = 0,4: expandir vale 20 kUM y piloto 27; información perfecta vale como máximo 14.

**Práctica:** Mover p a 0,6 y contrastar valor esperado, peor caso y preferencia del decisor.

**Criterio de logro:** Entregar matriz de pagos, alternativa, sensibilidad y cota del costo de información.

[Guía y secuencia](../guia_maestra_aed.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/AED_manual_cientifico.html#sec-decision)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
RAIZ=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'_transversal/datos').is_dir())
sys.path.insert(0,str(RAIZ/'04_Analitica_Estrategica_Datos/reproducibilidad'))
from aed_recursos import (casapeumo, indicadores, boldonet, plan_produccion,
                         decision, solar, resumen_solar, pregunta, DATOS, SEMILLA)
plt.rcParams.update({'figure.figsize':(8,3.5),'axes.spines.top':False,'axes.spines.right':False})

<a id="matriz"></a>
## 1. Acciones y estados de naturaleza

Caso docente, pagos netos en miles de UM: no actuar paga [0,0], expandir [-20,80] y piloto [15,45]
en demanda baja/alta. La probabilidad alta es un supuesto de escenario, no una frecuencia estimada.
La matriz contiene pagos **netos**; no reste otra vez la inversión. El valor esperado supone
neutralidad al riesgo y pagos comparables en horizonte y moneda.

$$EV(a)=\sum_s p_s V(a,s).$$

Un árbol de decisión representa secuencia de acciones y azar; no es un árbol predictivo CART.
Se resuelve hacia atrás: promedio ponderado en azar y selección en decisión. Al introducir una
señal se usan probabilidades condicionadas, evitando reutilizar las probabilidades iniciales.

In [2]:
from aed_recursos import PAGOS, ALTERNATIVAS
ev,evpi=decision(.4)
display(pd.DataFrame({'acción':ALTERNATIVAS,'pago_bajo':PAGOS[:,0],'pago_alto':PAGOS[:,1],'EV':ev}))
assert np.allclose(ev,[0,20,27]) and np.isclose(evpi,14)
@widgets.interact(p_alta=widgets.FloatSlider(value=.4,min=0,max=1,step=.05))
def comparar(p_alta=.4):
    valores,info=decision(p_alta)
    print('Mejor por EV:',ALTERNATIVAS[int(np.argmax(valores))],'; EVPI:',round(info,2))
    fig,ax=plt.subplots();ax.bar(ALTERNATIVAS,valores,color='#176b78');ax.axhline(0,color='grey');ax.set(ylabel='Valor esperado (kUM)');plt.show();plt.close(fig)
    assert info>=-1e-10

,acción,pago_bajo,pago_alto,EV
0,No actuar,0.0,0.0,0.0
1,Expandir,-20.0,80.0,20.0
2,Piloto,15.0,45.0,27.0


interactive(children=(FloatSlider(value=0.4, description='p_alta', max=1.0, step=0.05), Output()), _dom_classe…

<a id="informacion"></a>
## 2. Información perfecta e imperfecta

$$EVPI=\sum_s p_s\max_a V(a,s)-\max_a\sum_s p_sV(a,s)\ge0.$$

EVPI es una cota superior del valor bruto de información para este modelo. No es el valor de
cualquier estudio: un estudio imperfecto puede valer menos y su costo debe descontarse. Con una
señal de sensibilidad y especificidad 0.8 calculamos posteriores por Bayes y el valor de actuar
según la señal. La calidad de la señal es otro supuesto que requiere evidencia.

In [3]:
p=.4; sensibilidad=.8; especificidad=.8
p_positivo=sensibilidad*p+(1-especificidad)*(1-p)
post_pos=sensibilidad*p/p_positivo
post_neg=(1-sensibilidad)*p/(1-p_positivo)
ev_pos,_=decision(post_pos);ev_neg,_=decision(post_neg)
valor_senal=p_positivo*ev_pos.max()+(1-p_positivo)*ev_neg.max()
evsi=valor_senal-ev.max()
display(pd.Series({'P_señal_positiva':p_positivo,'P_alta_dado_positivo':post_pos,
 'P_alta_dado_negativo':post_neg,'EVSI_bruto':evsi,'EVPI':evpi,'valor_neto_si_cuesta_3':evsi-3}))
assert -1e-10<=evsi<=evpi+1e-10

P_señal_positiva           0.440000
P_alta_dado_positivo       0.727273
P_alta_dado_negativo       0.142857
EVSI_bruto                 7.000000
EVPI                      14.000000
valor_neto_si_cuesta_3     4.000000
dtype: float64

<a id="riesgo"></a>
## 3. Riesgo, arrepentimiento y restricciones

Maximin elige el mayor peor pago; minimax regret minimiza la mayor oportunidad perdida por
estado. No son equivalentes al valor esperado. Una restricción de pérdida máxima puede excluir
expandir aunque tenga EV alto. El criterio es una preferencia del decisor y debe declararse,
no elegirse después de mirar cuál favorece la recomendación.

In [4]:
arrepentimiento=PAGOS.max(axis=0)-PAGOS
display(pd.DataFrame({'acción':ALTERNATIVAS,'peor_pago':PAGOS.min(axis=1),'peor_arrepentimiento':arrepentimiento.max(axis=1)}))
assert np.argmax(PAGOS.min(axis=1))==2
assert np.argmin(arrepentimiento.max(axis=1))==1
print('Con restricción de no perder dinero, expansión queda fuera.')

,acción,peor_pago,peor_arrepentimiento
0,No actuar,0.0,80.0
1,Expandir,-20.0,35.0
2,Piloto,15.0,35.0


Con restricción de no perder dinero, expansión queda fuera.


<a id="multicriterio"></a>
## 4. Matriz multicriterio con preferencias explícitas

Cuando importan viabilidad, plazo e impacto, una suma ponderada exige escalas comparables y
preferencias compensatorias. Un requisito obligatorio se trata como restricción antes de puntuar:
una buena nota no compensa ilegalidad o presupuesto imposible. Aquí las puntuaciones 0–1 son
supuestos docentes, no mediciones monetarias. El ranking depende de pesos y anclajes; explore
su estabilidad y evite contar dos veces criterios muy correlacionados.

In [5]:
puntajes=np.array([[.2,.9],[.9,.3],[.6,.8]]) # impacto y facilidad, ambos mayor es mejor.
@widgets.interact(peso_impacto=widgets.FloatSlider(value=.5,min=0,max=1,step=.05))
def multicriterio(peso_impacto=.5):
    pesos=np.array([peso_impacto,1-peso_impacto]);resultado=puntajes@pesos
    display(pd.DataFrame({'acción':ALTERNATIVAS,'puntaje':resultado}))
    assert np.isclose(pesos.sum(),1) and np.all((resultado>=0)&(resultado<=1))

interactive(children=(FloatSlider(value=0.5, description='peso_impacto', max=1.0, step=0.05), Output()), _dom_…

<a id="ejercicio"></a>
## 5. Ejercicio resuelto

Expansión tiene $EV=-20+100p$ y piloto $EV=15+30p$. Igualando obtenemos $p=0.5$.
Por debajo conviene piloto bajo neutralidad al riesgo; por encima expansión. No actuar está
dominado por piloto en esta matriz, pero puede recuperar sentido si cambian costos o capacidad.
Solicite al decisor un rango plausible de $p$ y calcule si cruza el umbral. Un resultado sensible
justifica más información o una decisión reversible; no justifica ocultar la alternativa.

In [6]:
umbral=(15+20)/(100-30)
e,_=decision(umbral)
assert np.isclose(umbral,.5) and np.isclose(e[1],e[2])
print('Umbral de indiferencia:',umbral)

Umbral de indiferencia: 0.5


<a id="autoevaluacion"></a>
## Autoevaluación
Seleccione y compruebe su respuesta.

In [7]:
auto_05=pregunta('¿Qué representa EVPI?', ['El precio mínimo que debe pagarse por un estudio', 'La cota superior del valor bruto de información perfecta en el modelo', 'La rentabilidad garantizada del proyecto'], 1, 'Compara decidir con conocimiento perfecto del estado y decidir antes de conocerlo; depende del modelo y sus probabilidades.')